# Train the G1 on Google Colab

Trains our tasks (`Flat`, `Ramp`, `Stairs`, `Open`, ...; see `training/README.md`) on a Colab GPU. Same code and commands as on a laptop, without Docker.

- **Runtime → Change runtime type → GPU** first (T4 is the free one).
- Everything a run writes goes to your Google Drive (`MyDrive/g1_training/logs`), so a disconnect loses at most the last 99 iterations. Run the cells again and training continues from the newest saved model.
- Colab runs the code that is **pushed** to GitHub. Commit and push your changes first.

This is MuJoCo Warp (mjlab) + RSL-RL, not the MJX + Brax stack of DeepMind's MJX tutorial notebook: the tasks, models and commands here are the ones the rest of the team uses.

In [ ]:
#@title 1. Settings
REPO = "University-of-Toronto-Humanoid-Robotics/humanoid"
BRANCH = "competition"  #@param {type: "string"}
DRIVE_FOLDER = "g1_training"  #@param {type: "string"}

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU: Runtime -> Change runtime type -> GPU")

In [ ]:
#@title 2. Get the code and install (about 3 minutes)
import os
from google.colab import drive

drive.mount("/content/drive")

# Private repository? Add a GitHub token as a Colab secret named GITHUB_TOKEN
# (key icon on the left, "Notebook access" on). A public one needs nothing.
token = ""
try:
  from google.colab import userdata
  token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
  pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"

%cd /content
if not os.path.isdir("humanoid"):
  !git clone -q --branch {BRANCH} --recurse-submodules {url} humanoid
else:
  !cd humanoid && git pull -q && git submodule update --init

# Training runs live on Drive (training/logs is not in git, so the clone has none).
logs = f"/content/drive/MyDrive/{DRIVE_FOLDER}/logs"
os.makedirs(logs, exist_ok=True)
if not os.path.lexists("/content/humanoid/training/logs"):
  os.symlink(logs, "/content/humanoid/training/logs")

# The versions our Docker image uses (docker/Dockerfile). Colab's own PyTorch is kept.
!pip install -q mjlab==1.2.0 mujoco-warp==3.5.0 mujoco==3.5.0 warp-lang==1.12.0 rsl-rl-lib==5.0.1 wandb

# Lets MuJoCo render videos on Colab's GPU (from DeepMind's MuJoCo tutorials).
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
  with open(icd, "w") as f:
    f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ["MUJOCO_GL"] = "egl"
%cd /content/humanoid/training

In [ ]:
#@title 3. Check that a task builds (about 2 minutes the first time: GPU code is compiled)
!python scripts/check_task.py Flat --num-envs 64 --steps 50 2>&1 | tail -12

## Train

Robots at once (`NUM_ENVS`). More robots = better learning per iteration, but more GPU memory and slower iterations. Past the GPU's memory, training gets about 10× slower instead of stopping, so stay under it.

| GPU | `Flat` | `Ramp`, `Stairs`, `Open`, `Competition`, `Rough` |
|---|---|---|
| T4 (16 GB, free) | 4096 | 2048 |
| L4 (24 GB) | 8192 | 4096 |
| A100 (40 GB) | 8192 | 8192 |

These come from memory measured on an 8 GB laptop GPU (see `training/README.md`), scaled to the Colab GPUs; they have not been timed on Colab. 8192 is as many as PPO needs here.

The plan: `Flat` for about 3,000 iterations (the base model), then `Open` starting from it (`FROM`), then `Competition`. Stopped or disconnected? Run this cell again with the same `RUN_NAME`: it continues from that run's newest model.

In [ ]:
#@title 4. Train
TASK = "Flat"  #@param ["Flat", "Ramp", "Stairs", "Open", "Rough", "VeryRough", "Competition", "Flat-Crouch", "Ramp-Crouch", "Stairs-Crouch", "Open-Crouch", "Rough-Crouch", "VeryRough-Crouch", "Competition-Crouch"]
NUM_ENVS = 4096  #@param {type: "integer"}
ITERATIONS = 3000  #@param {type: "integer"}
RUN_NAME = "base"  #@param {type: "string"}
#@markdown Start from another run's model (a path under `logs/`, e.g. `logs/rsl_rl/flat/<folder>/model_2999.pt`). Leave empty to start from scratch.
FROM = ""  #@param {type: "string"}

import glob, re

def newest_model(task, run_name):
  """Newest saved model of this task's runs with this name, and its iteration."""
  folder = task.lower().replace("-", "_")
  models = glob.glob(f"logs/rsl_rl/{folder}/*_{run_name}/model_*.pt")
  if not models:
    return None, 0
  best = max(models, key=lambda p: int(re.search(r"model_(\d+)\.pt$", p).group(1)))
  return best, int(re.search(r"model_(\d+)\.pt$", best).group(1))

resume, done = newest_model(TASK, RUN_NAME)
start = resume or FROM
left = ITERATIONS - done if resume else ITERATIONS
if left <= 0:
  print(f"{RUN_NAME} already reached {done} iterations: {resume}")
else:
  print(f"{'Continuing ' + start if start else 'Starting from scratch'}; {left} iterations to go")
  from_option = f"--from {start}" if start else ""
  !python scripts/train.py {TASK} {from_option} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {left} --agent.logger tensorboard --agent.run-name {RUN_NAME} 2>&1 | grep --line-buffered -E "Learning iteration|Mean reward|Mean episode length|track_linear_velocity|terrain_levels|Iteration time|Error|Traceback|Continuing"

In [ ]:
#@title 5. Learning curves (run any time; training can be stopped or running in another session)
%load_ext tensorboard
%tensorboard --logdir logs/rsl_rl

In [ ]:
#@title 6. Watch the newest model (records a 20 s video of it walking forward)
WATCH_ON = "Flat"  #@param ["Flat", "Ramp", "Stairs", "Open", "Rough", "VeryRough", "Competition"]
FIRST_PERSON = False  #@param {type: "boolean"}

from IPython.display import Video
model, _ = newest_model(TASK, RUN_NAME)
fpv = "--fpv" if FIRST_PERSON else ""
!python scripts/record_video.py {WATCH_ON} --checkpoint {model} {fpv} 2>&1 | tail -1
Video(f"logs/videos/{WATCH_ON}{'_fpv' if FIRST_PERSON else ''}.mp4", embed=True)

## Is it working?

- **Mean episode length** should climb to 1000 (20 s: the robot no longer falls). On `Flat` that takes about 300-500 iterations.
- **track_linear_velocity** should climb toward 0.7-0.8: it follows the commanded speed.
- **terrain_levels** (not on `Flat`) climbs as robots handle harder rows (0-9).

To use a model on a laptop, download its `model_<N>.pt` from Drive (`g1_training/logs/rsl_rl/<task>/<run folder>/`) into `training/models/` and play it there: `python scripts/play.py Open --viewer viser --checkpoint-file models/<file>`.